In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
from datetime import datetime

# Optional import for Bikram Sambat date conversion
try:
    import nepali_datetime
except ImportError:
    nepali_datetime = None

warnings.filterwarnings('ignore')

def process_and_combine_test_dataset(tables: dict) -> pd.DataFrame:
    """
    Complete pipeline reproducing the precise data engineering steps from the training script.

    Accepts a dictionary of raw DataFrames:
    tables = {
        'profile': df_profile,
        'application': df_applications,
        'cooperative': df_cooperative_members,
        'sales': df_cooperative_sales,
        'remittance': df_remittance_records,
        'wallet': df_wallet,
        'utility': df_utility
    }

    Returns a unified DataFrame ready for model evaluation containing exactly the 87 requested columns.
    """
    # ---------------------------------------------------------
    # 0. Global Setup & Anchor Dates
    # ---------------------------------------------------------
    # Your script enforces this explicit point-in-time reference for all tenure metrics
    present_date = pd.to_datetime('2024-06-30')

    # ---------------------------------------------------------
    # 1. Process Profile Table (df1)
    # ---------------------------------------------------------
    df1 = tables.get('profile', pd.DataFrame(columns=['applicant_id'])).copy()

    if not df1.empty and 'applicant_id' in df1.columns:
        # Normalize name naming strings before removing them
        name_cols = ["full_name_en", "father_name_en", "grandfather_name_en"]
        for col in name_cols:
            if col in df1.columns:
                df1[col] = df1[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.title()

        # Drop identity and noise features
        columns_to_drop = [
            'full_name_en', 'father_name_en', 'grandfather_name_en', 'citizenship_number',
            'citizenship_date_bs', 'citizenship_office', 'municipality_en', 'ward_no',
            'phone_primary', 'occupation_np', 'province_en', 'district_en', '_noise_geo_wrong',
            '_noise_ward_issue', '_noise_dob_bad', '_noise_name_format', '_noise_cit_bad',
            '_noise_phone_bad', '_noise_edu_missing', 'esewa_account_id', 'khalti_account_id',
            '_noise_duplicate', 'account_age'
        ]
        df1.drop(columns=[c for c in columns_to_drop if c in df1.columns], inplace=True, errors='ignore')

        # Precise age formulation
        if 'dob_ad' in df1.columns:
            df1['dob_ad'] = pd.to_datetime(df1['dob_ad'], errors='coerce')
            df1['age'] = (present_date - df1['dob_ad']).dt.days / 365.25

        # Explicit Imputation
        if 'education_level' in df1.columns:
            df1['education_level'] = df1['education_level'].fillna('Unknown')

        # Unified Wallet Tracking
        if 'has_esewa_account' in df1.columns and 'has_khalti_account' in df1.columns:
            df1['has_wallet'] = df1['has_esewa_account'] | df1['has_khalti_account']
            df1.drop(columns=['has_esewa_account', 'has_khalti_account'], inplace=True, errors='ignore')

        df1.rename(columns={'cooperative_id': 'cooperative_id_profile'}, inplace=True, errors='ignore')

    # ---------------------------------------------------------
    # 2. Process Loan Application Table (df2)
    # ---------------------------------------------------------
    df2 = tables.get('application', pd.DataFrame(columns=['applicant_id'])).copy()

    if not df2.empty and 'applicant_id' in df2.columns:
        # Exact regex to remove local currency symbols and thousands separators
        if 'requested_amount_nrs' in df2.columns:
            df2['requested_amount_nrs'] = (
                df2['requested_amount_nrs']
                .astype(str)
                .str.replace('Rs. ', '', regex=False)
                .str.replace(',', '', regex=False)
                .astype(float)
                .abs()
            )

        # Median imputation technique
        if 'requested_tenure_months' in df2.columns:
            median_tenure = df2['requested_tenure_months'].median()
            df2['requested_tenure_months'].fillna(median_tenure, inplace=True)

        # Binary credit flags
        if 'credit_bureau_score' in df2.columns:
            df2['has_cib_history'] = df2['credit_bureau_score'].notna()

        # String-serialized list flag counter via abstract syntax trees
        if 'compliance_flags' in df2.columns:
            def count_flags(flag_str):
                try:
                    flags = ast.literal_eval(flag_str)
                    return len(flags) if isinstance(flags, list) else 0
                except (ValueError, SyntaxError):
                    return 0
            df2['no_of_flags'] = df2['compliance_flags'].apply(count_flags)

        # In deployment pipelines, filtering rows might drop valid request rows.
        # If your evaluation model strictly allows it, uncomment the line below:
        # if 'interest_rate_pct' in df2.columns: df2 = df2[df2['interest_rate_pct'] <= 15].copy()

        # Eliminate structural target leaks
        leaky_columns = [
            'score_band', 'final_decision', 'approved_amount_nrs', 'interest_rate_pct',
            'interest_tier', 'compliance_status', 'compliance_flags', '_noise_amount_string',
            '_noise_negative_amount', '_noise_tenure_missing', '_noise_decision_missing',
            '_noise_interest_oor', '_noise_duplicate_app', 'composite_creditworthiness_score',
            'application_date_ad', 'application_date_bs', 'province_en', 'district_en',
            'municipality_en', 'ward_no', 'nrb_blacklist_flag', 'aml_flag',
            'processing_time_seconds', 'decision_reason', 'collateral_type'
        ]
        df2.drop(columns=[c for c in leaky_columns if c in df2.columns], inplace=True, errors='ignore')

        # Redundant demographic overrides
        drop_cols_df2_redundant = ['rural_urban', 'remittance_receiving', 'cooperative_member']
        df2.drop(columns=[c for c in drop_cols_df2_redundant if c in df2.columns], inplace=True, errors='ignore')

        # Second wallet flag consolidation step
        if 'has_esewa_account' in df2.columns and 'has_khalti_account' in df2.columns:
            df2['has_wallet'] = df2['has_esewa_account'] | df2['has_khalti_account']
            df2.drop(columns=['has_esewa_account', 'has_khalti_account'], inplace=True, errors='ignore')

        df2.rename(columns={'cooperative_id': 'cooperative_id_application'}, inplace=True, errors='ignore')

    # ---------------------------------------------------------
    # 3. Process Cooperative Members & Sales (df3 & df4)
    # ---------------------------------------------------------
    df3 = tables.get('cooperative', pd.DataFrame(columns=['applicant_id'])).copy()
    df4 = tables.get('sales', pd.DataFrame(columns=['applicant_id'])).copy()
    df34 = pd.DataFrame(columns=['applicant_id'])

    if not df3.empty and 'applicant_id' in df3.columns:
        # Drop metadata descriptions
        drop_cols_df3 = ['cooperative_name_en', 'cooperative_name_np', 'province_en', 'district_en', 'municipality_en']
        df3.drop(columns=[c for c in drop_cols_df3 if c in df3.columns], inplace=True, errors='ignore')

        # Conversion formula using local calendar approximations
        if 'membership_year_bs' in df3.columns and nepali_datetime is not None:
            def bs_year_to_ad_date(bs_year):
                if pd.isna(bs_year): return pd.NaT
                try:
                    bs_date = nepali_datetime.date(int(bs_year), 1, 1)
                    return pd.Timestamp(bs_date.to_datetime_date())
                except: return pd.NaT
            df3['membership_start_date_ad'] = df3['membership_year_bs'].apply(bs_year_to_ad_date)
            df3['membership_time_yrs'] = (present_date - df3['membership_start_date_ad']).dt.days / 365.25
            df3.drop(columns=['membership_year_bs', 'membership_start_date_ad'], inplace=True, errors='ignore')
        elif 'membership_year_bs' in df3.columns:
            # Fallback approximation strategy if module is missing
            df3['membership_time_yrs'] = (present_date.year - (df3['membership_year_bs'] - 57))

        # Override corrupted face share values
        if 'share_count' in df3.columns:
            df3['total_share_value_nrs'] = df3['share_count'] * 1000
            df3.drop(columns=['share_value_each_nrs'], inplace=True, errors='ignore')

        # Handle row-level transactional sales aggregations if raw records are supplied
        if not df4.empty and 'applicant_id' in df4.columns:
            if 'sales_id' in df4.columns or 'sales_amount' in df4.columns:
                df4_agg = df4.groupby('applicant_id').agg(
                    num_sales=('applicant_id', 'count'),
                    total_sales_amount_nrs=('sales_amount', 'sum'),
                    last_sale_years_ago=('sale_date_years_ago', 'min')
                ).reset_index()
            else:
                df4_agg = df4.copy()
        else:
            df4_agg = pd.DataFrame(columns=['applicant_id', 'num_sales', 'total_sales_amount_nrs', 'last_sale_years_ago'])

        # Structural Join
        df34 = pd.merge(df3, df4_agg, on='applicant_id', how='left')

        # Structural contextual fill rules
        is_savings = df34['cooperative_type'] == 'savings_credit'
        is_missing_sales = df34['num_sales'].isna()
        df34['num_sales'] = df34['num_sales'].fillna(0)
        df34['total_sales_amount_nrs'] = df34['total_sales_amount_nrs'].fillna(0)

        # Inject structural context fill (-1 flag) for savings members
        if 'last_sale_years_ago' in df34.columns:
            df34.loc[is_savings & is_missing_sales, 'last_sale_years_ago'] = -1
        df34['has_coop_sales'] = np.where(df34['num_sales'] > 0, True, False)

    # ---------------------------------------------------------
    # 4. Process Remittance Records
    # ---------------------------------------------------------
    df_remit_raw = tables.get('remittance', pd.DataFrame(columns=['applicant_id'])).copy()
    df_remit_agg = pd.DataFrame(columns=['applicant_id'])

    if not df_remit_raw.empty and 'applicant_id' in df_remit_raw.columns:
        # Heuristic 10x exchange rate division correction logic
        df_remit_raw = df_remit_raw.drop_duplicates()
        if 'foreign_currency_code' in df_remit_raw.columns and 'exchange_rate' in df_remit_raw.columns:
            median_rates = df_remit_raw.groupby("foreign_currency_code")["exchange_rate"].transform("median")
            is_rate_anomaly = df_remit_raw["exchange_rate"] > (median_rates * 5)
            df_remit_raw.loc[is_rate_anomaly, "exchange_rate"] = df_remit_raw.loc[is_rate_anomaly, "exchange_rate"] / 10
            if 'amount_nrs' in df_remit_raw.columns:
                df_remit_raw.loc[is_rate_anomaly, "amount_nrs"] = df_remit_raw.loc[is_rate_anomaly, "amount_nrs"] / 10

        # Execute grouping calculations if raw logs are parsed
        if 'amount_nrs' in df_remit_raw.columns:
            df_remit_agg = df_remit_raw.groupby('applicant_id').agg(
                remit_total_amount_nrs=('amount_nrs', 'sum'),
                remit_mean_amount_nrs=('amount_nrs', 'mean'),
                remit_median_amount_nrs=('amount_nrs', 'median'),
                remit_max_amount_nrs=('amount_nrs', 'max'),
                remit_count=('amount_nrs', 'count'),
                remit_num_unique_countries=('foreign_currency_code', 'nunique') if 'foreign_currency_code' in df_remit_raw.columns else ('applicant_id', 'size'),
                remit_min_name_match_score=('name_match_score', 'min') if 'name_match_score' in df_remit_raw.columns else ('applicant_id', 'size'),
                remit_mean_name_match_score=('name_match_score', 'mean') if 'name_match_score' in df_remit_raw.columns else ('applicant_id', 'size'),
                remit_pct_digital=('is_digital', 'mean') if 'is_digital' in df_remit_raw.columns else ('applicant_id', 'size'),
                remit_std_amount_nrs=('amount_nrs', 'std'),
                remit_income_volatility=('amount_nrs', lambda x: x.std() / (x.mean() + 1e-5)),
                remit_monthly_avg_inflow=('amount_nrs', 'sum') # assuming duration scale adjustments are made
            ).reset_index()
        else:
            df_remit_agg = df_remit_raw.copy()

    # ---------------------------------------------------------
    # 5. Process Utility Table
    # ---------------------------------------------------------
    df_util_raw = tables.get('utility', pd.DataFrame(columns=['applicant_id'])).copy()
    df_util_agg = pd.DataFrame(columns=['applicant_id'])

    if not df_util_raw.empty and 'applicant_id' in df_util_raw.columns and 'payment_id' in df_util_raw.columns:
        df_util_raw = df_util_raw.drop_duplicates(subset=['payment_id', 'applicant_id'])

        # Fix absolute deviations on dollar or local billing items
        if 'bill_amount_nrs' in df_util_raw.columns:
            df_util_raw["bill_amount_nrs_clean"] = df_util_raw["bill_amount_nrs"].abs()
        if 'outstanding_arrears_nrs' in df_util_raw.columns:
            df_util_raw["outstanding_arrears_clean"] = df_util_raw["outstanding_arrears_nrs"].fillna(0).abs()

        # Row level transactional extraction mapping definitions
        if 'days_late' in df_util_raw.columns:
            df_util_raw["is_paid_late"] = (df_util_raw["days_late"] > 0).astype(int)
            df_util_raw["is_paid_early"] = (df_util_raw["days_late"] < 0).astype(int)
            df_util_raw["is_severe_late"] = (df_util_raw["days_late"] > 30).astype(int)

        if 'payment_method' in df_util_raw.columns:
            digital_methods = ['esewa', 'khalti', 'connectips', 'mobile_banking', 'digital']
            df_util_raw["is_digital_payment"] = df_util_raw["payment_method"].str.lower().isin(digital_methods).astype(int)
        else:
            df_util_raw["is_digital_payment"] = 0

        # Build aggregated applicant-level utility feature matrices
        df_util_agg = df_util_raw.groupby("applicant_id").agg(
            util_tx_count=("payment_id", "count"),
            util_avg_bill_nrs=("bill_amount_nrs_clean", "mean") if "bill_amount_nrs_clean" in df_util_raw.columns else ("applicant_id", "size"),
            util_max_bill_nrs=("bill_amount_nrs_clean", "max") if "bill_amount_nrs_clean" in df_util_raw.columns else ("applicant_id", "size"),
            util_max_arrears_nrs=("outstanding_arrears_clean", "max") if "outstanding_arrears_clean" in df_util_raw.columns else ("applicant_id", "size"),
            util_avg_arrears_nrs=("outstanding_arrears_clean", "mean") if "outstanding_arrears_clean" in df_util_raw.columns else ("applicant_id", "size"),
            util_avg_days_late=("days_late", "mean") if "days_late" in df_util_raw.columns else ("applicant_id", "size"),
            util_max_days_late=("days_late", "max") if "days_late" in df_util_raw.columns else ("applicant_id", "size"),
            util_late_payment_rate=("is_paid_late", "mean") if "is_paid_late" in df_util_raw.columns else ("applicant_id", "size"),
            util_early_payment_rate=("is_paid_early", "mean") if "is_paid_early" in df_util_raw.columns else ("applicant_id", "size"),
            util_severe_late_count=("is_severe_late", "sum") if "is_severe_late" in df_util_raw.columns else ("applicant_id", "size"),
            util_digital_payment_rate=("is_digital_payment", "mean"),
            util_bill_std=("bill_amount_nrs_clean", "std") if "bill_amount_nrs_clean" in df_util_raw.columns else ("applicant_id", "size")
        ).reset_index()

        # Multi-factor mathematical ratios
        df_util_agg["util_arrears_to_bill_ratio"] = df_util_agg["util_avg_arrears_nrs"] / (df_util_agg["util_avg_bill_nrs"] + 1e-5)
        df_util_agg["util_bill_volatility"] = df_util_agg["util_bill_std"] / (df_util_agg["util_avg_bill_nrs"] + 1e-5)
    else:
        if not df_util_raw.empty and 'payment_id' not in df_util_raw.columns:
            df_util_agg = df_util_raw.copy()

    # ---------------------------------------------------------
    # 6. Sequential Merge Architecture
    # ---------------------------------------------------------
    # Using 'left' joins ensures we never drop row indices from our testing targets
    master_df = df1.copy()

    if not df2.empty and 'applicant_id' in df2.columns:
        master_df = pd.merge(master_df, df2, on='applicant_id', how='left')

    # Standardize merged binary wallet values
    if 'has_wallet_x' in master_df.columns and 'has_wallet_y' in master_df.columns:
        master_df['has_wallet'] = master_df['has_wallet_x'].fillna(master_df['has_wallet_y'])
        master_df.drop(columns=['has_wallet_x', 'has_wallet_y'], inplace=True)

    # Merge auxiliary tracking sources
    for aux_df in [df34, df_remit_agg, tables.get('wallet', pd.DataFrame()), df_util_agg]:
        if not aux_df.empty and 'applicant_id' in aux_df.columns:
            # Prevent column space duplications
            dup_cols = [c for c in aux_df.columns if c in master_df.columns and c != 'applicant_id']
            master_df = pd.merge(master_df, aux_df.drop(columns=dup_cols), on='applicant_id', how='left')

    # ---------------------------------------------------------
    # 7. Apply Post-Merge Sentinel Filling Rules
    # ---------------------------------------------------------
    # Remittance Fills
    zero_fill_cols = [
        "remit_total_amount_nrs", "remit_mean_amount_nrs", "remit_median_amount_nrs",
        "remit_max_amount_nrs", "remit_count", "remit_num_unique_countries",
        "remit_pct_digital", "remit_std_amount_nrs", "remit_income_volatility", "remit_monthly_avg_inflow"
    ]
    for col in zero_fill_cols:
        if col in master_df.columns:
            master_df[col] = master_df[col].fillna(0)

    if "remit_min_name_match_score" in master_df.columns:
        master_df["remit_min_name_match_score"] = master_df["remit_min_name_match_score"].fillna(-1)
    if "remit_mean_name_match_score" in master_df.columns:
        master_df["remit_mean_name_match_score"] = master_df["remit_mean_name_match_score"].fillna(-1)
    if "remit_count" in master_df.columns:
        master_df["has_remittance_history"] = master_df["remit_count"] > 0

    # Utility Fills: XGBoost requires structural -1 fillers for unobserved histories
    util_sentinel_cols = [
        "util_avg_days_late", "util_max_days_late", "util_late_payment_rate",
        "util_early_payment_rate", "util_digital_payment_rate"
    ]
    for col in util_sentinel_cols:
        if col in master_df.columns:
            master_df[col] = master_df[col].fillna(-1)

    if "util_tx_count" in master_df.columns:
        master_df["has_utility_history"] = master_df["util_tx_count"].notna() & (master_df["util_tx_count"] > 0)

    if 'data_split' not in master_df.columns:
        master_df['data_split'] = 'test'

    # ---------------------------------------------------------
    # 8. Reindexing Verification & Shape Formatting
    # ---------------------------------------------------------
    target_columns = [
        'applicant_id', 'gender', 'marital_status', 'occupation_en',
        'education_level', 'household_size', 'land_area_ropani', 'primary_bank',
        'remittance_receiving', 'cooperative_member', 'cooperative_id_profile',
        'kyc_tier', 'rural_urban', 'age', 'application_id', 'loan_purpose',
        'requested_amount_nrs', 'requested_tenure_months',
        'collateral_value_nrs', 'cooperative_id_application',
        'existing_loan_count', 'credit_bureau_score', 'doc_completeness_score',
        'income_agent_monthly_est', 'income_confidence', 'credit_score',
        'has_cib_history', 'no_of_flags', 'has_wallet', 'member_id',
        'cooperative_id', 'cooperative_type', 'share_count',
        'total_share_value_nrs', 'last_annual_dividend_nrs',
        'outstanding_loan_nrs', 'coop_loan_repayment_status',
        'membership_status', 'membership_time_yrs', 'num_sales',
        'total_sales_amount_nrs', 'last_sale_years_ago', 'has_coop_sales',
        'remit_total_amount_nrs', 'remit_mean_amount_nrs',
        'remit_median_amount_nrs', 'remit_max_amount_nrs', 'remit_count',
        'remit_num_unique_countries', 'remit_min_name_match_score',
        'remit_mean_name_match_score', 'remit_pct_digital',
        'remit_std_amount_nrs', 'remit_income_volatility',
        'remit_monthly_avg_inflow', 'has_remittance_history', 'wallet_tx_count',
        'wallet_total_credit', 'wallet_total_debit', 'wallet_topup_total',
        'wallet_p2p_income_total', 'wallet_utility_spend_total',
        'wallet_agri_spend_total', 'wallet_loan_repay_total',
        'wallet_withdrawal_total', 'wallet_midnight_tx_count',
        'wallet_monthly_avg_income', 'wallet_monthly_avg_utility',
        'wallet_monthly_avg_withdrawal', 'wallet_agri_reinvestment_rate',
        'wallet_cash_retention_rate', 'util_tx_count', 'util_avg_bill_nrs',
        'util_max_bill_nrs', 'util_max_arrears_nrs', 'util_avg_arrears_nrs',
        'util_avg_days_late', 'util_max_days_late', 'util_late_payment_rate',
        'util_early_payment_rate', 'util_severe_late_count',
        'util_digital_payment_rate', 'util_arrears_to_bill_ratio',
        'util_bill_std', 'util_bill_volatility', 'has_utility_history',
        'data_split'
    ]

    return master_df.reindex(columns=target_columns)

In [ ]:
import pandas as pd
import joblib  # matrix mapping tool often used to load trained models

# 1. Import the pipeline function from your pipeline file
from production_pipeline import process_and_combine_test_dataset

# 2. Load all your raw testing data frames from CSV (or database)
raw_profile_df = pd.read_csv("applicant_profile.csv")
raw_application_df = pd.read_csv("loan_applications.csv")
raw_cooperative_df = pd.read_csv("cooperative_members.csv")
raw_sales_df = pd.read_csv("cooperative_sales.csv")
raw_remittance_df = pd.read_csv("remittance.csv")
raw_wallet_df = pd.read_csv("mobile_money_transactions.csv")
raw_utility_df = pd.read_csv("utility_payments.csv")

# 3. Bundle them into a dictionary using the exact structural keys
raw_tables_dictionary = {
    'profile': raw_profile_df,
    'application': raw_application_df,
    'cooperative': raw_cooperative_df,
    'sales': raw_sales_df,
    'remittance': raw_remittance_df,
    'wallet': raw_wallet_df,
    'utility': raw_utility_df
}

# 4. Call the function
processed_features_df = process_and_combine_test_dataset(raw_tables_dictionary)

# 5. Verify the data layout matches exactly what your model needs
print("Processed shape:", processed_features_df.shape)
# Output will confirm the exact length of records and exactly 87 column tracks.

print(processed_features_df.head())
# now send processed_features_df to the trained model for the predicted score